In [18]:
#pip install pandas


In [4]:
# Este código prepara la base de datos para el taller.
import pandas as pd
import requests
import sqlite3

C:\Users\jesus\anaconda3\envs\izainea_env\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [26]:
import pandas as pd
import requests
import sqlite3

# 1. Cargar datos desde API de datos.gov.co
api_url = "https://www.datos.gov.co/resource/d7zw-hpf4.json?$limit=10000"
response = requests.get(api_url)
data = response.json()

# 2. Convertir a DataFrame
df_secuestros = pd.DataFrame(data)

# Datos de población (ejemplo)
poblacion_data = {
    'nombre_depto': ['ANTIOQUIA', 'CUNDINAMARCA', 'VALLE DEL CAUCA', 'ATLÁNTICO', 'BOGOTÁ, D.C.', 'SANTANDER', 'BOLÍVAR'],
    'poblacion': [6677930, 3242996, 4532152, 2722128, 7743955, 2280908, 2180976]
}
df_poblacion = pd.DataFrame(poblacion_data)

# Crear y poblar la base de datos
conn = sqlite3.connect('taller_seguridad.db')
df_secuestros.to_sql('secuestros', conn, index=False, if_exists='replace')
df_poblacion.to_sql('poblacion_deptos', conn, index=False, if_exists='replace')
conn.close()

print("Base de datos 'taller_seguridad.db' lista para el taller.")

Base de datos 'taller_seguridad.db' lista para el taller.


In [25]:
url = "https://www.datos.gov.co/resource/d7zw-hpf4.json?$limit=10000"
response = requests.get(url)
data = response.json()
df = pd.DataFrame(data)

# Limpiar y convertir columnas
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce').fillna(0).astype(int)
df['fecha_hecho'] = pd.to_datetime(df['fecha_hecho'], errors='coerce')

# Crear base de datos y guardar datos
conn = sqlite3.connect('secuestros.db')
df.to_sql('secuestros', conn, index=False, if_exists='replace')

# Consulta 1: total por departamento
consulta_1 = """
SELECT departamento, SUM(cantidad) AS total_secuestros
FROM secuestros
GROUP BY departamento
ORDER BY total_secuestros DESC
"""
df_1 = pd.read_sql_query(consulta_1, conn)
print("Total por departamento:")
print(df_1.head())

# Consulta 2: total por año
consulta_2 = """
SELECT strftime('%Y', fecha_hecho) AS año, SUM(cantidad) AS total
FROM secuestros
GROUP BY año
ORDER BY año
"""
df_2 = pd.read_sql_query(consulta_2, conn)
print("Total por año:")
print(df_2)

# Consulta 3: top 5 municipios
consulta_3 = """
SELECT municipio, SUM(cantidad) AS total
FROM secuestros
GROUP BY municipio
ORDER BY total DESC
LIMIT 5
"""
df_3 = pd.read_sql_query(consulta_3, conn)
print("Top 5 municipios con más secuestros:")
print(df_3)

# Cerrar la conexión (¡al final!)
conn.close()



Total por departamento:
         departamento  total_secuestros
0           ANTIOQUIA              1992
1               CESAR               833
2           SANTANDER               737
3     VALLE DEL CAUCA               686
4  NORTE DE SANTANDER               587
Total por año:
    año  total
0  1996   1038
1  1997   1624
2  1998   2860
3  1999   3205
4  2000   1273
Top 5 municipios con más secuestros:
     municipio  total
0  BOGOTA D.C.    433
1         CALI    412
2     MEDELLIN    372
3   VALLEDUPAR    199
4  BUCARAMANGA    184
